# 🤖 شام — تجربة البوت (sham_bot_test)

يحلّ محل خلية تجربة البوت القديمة التي كانت تشغّل نقطة حفظ **نوفا** القديمة بدل شام.

يجلب تلقائياً (بلا أي «Add Input») أحدث نموذج لشام:
1. ناتج **المرحلة الثانية** `sham-multimodal-checkpoint` (نص + صورة + صوت) إن نُشر.
2. وإلا آخر نقطة حفظ من **المرحلة الأولى** `sham-checkpoint` (نص فقط).

ويستخدم دائماً أداة تقسيم النص المحفوظة مع نفس النقطة، ثم يجيب على أسئلة تجريبية بالعربية.

**الإعداد:** Accelerator: None (يكفي المعالج العادي؛ GPU أسرع). Internet: On. Secrets: `GITHUB_TOKEN`، `KAGGLE_USERNAME`، `KAGGLE_KEY`.

### 1) سحب كود شام من GitHub

In [ ]:
import os
import sys
import subprocess
from kaggle_secrets import UserSecretsClient

GITHUB_TOKEN = UserSecretsClient().get_secret("GITHUB_TOKEN")
REPO_URL = f"https://{GITHUB_TOKEN}@github.com/jonsnow-org/Ttbik.git"
BRANCH = "claude/free-services-marketplace-h6rwk2"
CLONE_DIR = "/kaggle/working/Ttbik"

if not os.path.exists(CLONE_DIR):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, CLONE_DIR], check=True)
else:
    subprocess.run(["git", "-C", CLONE_DIR, "pull"], check=True)

# أمان: git يحفظ رابط الاستنساخ (بما فيه GITHUB_TOKEN) حرفياً داخل
# .git/config -- وهذا المجلد يبقى ضمن نتاج (Output) هذه الجلسة، الذي قد
# يُستخدَم لاحقاً كمُدخَل (Notebook Output) لجلسة أخرى، أو يُشارَك بأي شكل.
# نزع التوكن من الرابط المحفوظ فور نجاح الاستنساخ يمنع تسربه عبر هذا
# المسار تماماً (ثغرة حقيقية اكتشفتها المالكة، 2026-09-21).
subprocess.run(["git", "-C", CLONE_DIR, "remote", "set-url", "origin", "https://github.com/jonsnow-org/Ttbik.git"], check=True)

CODE_DIR = os.path.join(CLONE_DIR, "ai-system", "colab", "sham_small")
assert os.path.exists(os.path.join(CODE_DIR, "model.py"))
sys.path.insert(0, CODE_DIR)
print("كود شام جاهز في:", CODE_DIR)

In [ ]:
try:
    import tokenizers
except ImportError:
    subprocess.run(["pip", "install", "-q", "tokenizers"], check=True)
print("tokenizers جاهزة.")

### 2) تحميل أحدث نموذج لشام (يُجلب تلقائياً)

In [ ]:
import torch
from checkpoint import load_checkpoint
from text_tokenizer import ShamTextTokenizer
from sham_inputs import fetch_dataset, resume_text_lineage

device = "cuda" if torch.cuda.is_available() else "cpu"
MULTIMODAL_FILE = "final_" + "multimodal.pt"

_own = fetch_dataset("sham-multimodal-checkpoint")
_ck = sorted(_own.rglob(MULTIMODAL_FILE)) if _own else []
_tk = sorted(_own.rglob("sham_small_tokenizer.json")) if _own else []
if _ck and _tk:
    ckpt_path, tok_path, source = _ck[0], _tk[0], "المرحلة الثانية (نص + صورة + صوت)"
else:
    _lin = resume_text_lineage("sham-checkpoint", tokenizer_pattern="sham_small_tokenizer.json")
    assert _lin, "لا توجد أي نقطة حفظ لشام بعد — شغّلي دفتر المرحلة الأولى أولاً."
    ckpt_path, tok_path, source = _lin["checkpoint"], _lin["tokenizer"], "المرحلة الأولى (نص فقط)"

model, step, _ = load_checkpoint(ckpt_path, map_location=device)
model.eval()
tokenizer = ShamTextTokenizer.load(str(tok_path))
print(f"✅ شام محمّل من {source}: {ckpt_path.name} — الخطوة {step:,}، "
      f"{model.count_parameters():,} معامل، vocab={tokenizer.vocab_size:,}، الجهاز {device}")

### 3) الرد على رسالة

In [ ]:
from generate import generate_tokens
from model import SpecialTokens

def sham_reply(prompt: str, max_new_tokens: int = 120, temperature: float = 0.8) -> str:
    prompt_ids = torch.tensor([tokenizer.encode(prompt)], dtype=torch.long, device=device)
    out = generate_tokens(
        model, prompt_ids, max_new_tokens=max_new_tokens,
        temperature=temperature, top_k=50, top_p=0.95, eos_id=SpecialTokens.EOS,
        # نص فقط: كل رقم يولّده النموذج يجب أن يكون له مقابل في أداة التقسيم نفسها.
        allowed_ranges=[(0, tokenizer.vocab_size)] * max_new_tokens,
    )
    ids = [i for i in out[0, prompt_ids.shape[1]:].tolist() if i != SpecialTokens.EOS]
    return tokenizer.decode(ids)

for q in ["دمشق هي", "ما هو الذكاء الاصطناعي؟", "اكتب جملة عن فوائد القراءة:"]:
    print(f"👤 {q}\n🤖 {sham_reply(q)}\n")

### 4) جرّبي رسالتك

غيّري النص أدناه وشغّلي الخلية.

In [ ]:
print(sham_reply("مرحباً شام، عرّفي بنفسك"))